# CrossWise — training pipeline on Kaggle

Runs the code from `ml/` on a Kaggle notebook. Push it from your own machine:

```bash
python scripts/kaggle_run.py run --mode smoke          # ~20 min, no external data: proves every script works
python scripts/kaggle_run.py run --mode full --gpu     # the real pipeline
```

`kaggle_run.py` uploads `ml/` as a private Kaggle dataset, rewrites the `CONFIG` line below, pushes this notebook,
waits for the run and downloads the outputs into `ml/kaggle_output/`. No Google Drive anywhere.

* **smoke**: synthetic dataset and video → merge → pseudo-label → train → auto-label → review round-trip → LiteRT export.
* **full**: Roboflow + ImVisible (+ any Kaggle datasets you attach) → v0 → ImVisible mining → v1 → evaluation → export.

Outputs land in `/kaggle/working/artifacts`; heavy intermediates stay in a scratch folder that Kaggle does not save.
Every command goes through `run()`, which raises on a non-zero exit code — IPython's `!` swallows exit codes, so a
broken step would otherwise leave the notebook "complete" with nothing to show for it.

In [ ]:
CONFIG = {"mode": "smoke", "epochs_v0": 60, "epochs_v1": 120, "imgsz": 640}  # kaggle_run.py rewrites this line

import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path

MODE = CONFIG["mode"]
WORK = Path("/kaggle/working")
ML, ARTIFACTS = WORK / "ml", WORK / "artifacts"


def run(command: str, allow_fail: bool = False) -> int:
    print(f"\n$ {command}", flush=True)
    start = time.time()
    code = subprocess.run(command, shell=True).returncode
    print(f"[exit {code} after {time.time() - start:.0f}s]", flush=True)
    if code and not allow_fail:
        raise SystemExit(f"FAILED ({code}): {command}")
    return code


ARTIFACT_PATTERNS = ("*.tflite", "runs/detect/*/weights/best.pt", "runs/detect/*/weights/*.tflite",
                     "runs/detect/*/*.json", "runs/*.json", "data/merged/meta/*.csv",
                     "data/raw/local_auto/review_summary.json", "data/raw/local_auto/autolabel_report.csv")
NOISE = {"runs", "detect", "data", "raw", "merged", "weights", "meta"}   # folders that say nothing about the file


def collect() -> list[str]:
    # Called after every expensive step: /kaggle/working/artifacts is all that survives, and a failure three
    # hours in must not throw away the model that was already trained.
    for pattern in ARTIFACT_PATTERNS:
        for path in glob.glob(pattern):
            parents = [part for part in Path(path).parent.parts if part not in NOISE]
            shutil.copy(path, ARTIFACTS / "_".join([*parents, Path(path).name]))
    return sorted(p.name for p in ARTIFACTS.iterdir())


def scratch_dir() -> Path:
    # First writable scratch place: /kaggle/temp does not exist on every Kaggle image.
    for candidate in (Path("/kaggle/temp"), Path("/tmp/crosswise"), WORK / "scratch"):
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            probe = candidate / ".write_test"
            probe.write_text("ok")
            probe.unlink()
            return candidate
        except OSError:
            continue
    raise SystemExit("No writable scratch directory found")


SCRATCH = scratch_dir()


def find_code() -> Path:
    # Datasets mount as /kaggle/input/datasets/<user>/<slug>/ on current images, so search a few levels.
    level = [Path("/kaggle/input")]
    for _ in range(4):
        children = []
        for parent in level:
            for child in sorted(p for p in parent.iterdir() if p.is_dir()) if parent.is_dir() else []:
                if (child / "scripts" / "train.py").exists():
                    return child
                children.append(child)
        level = children
    for archive in sorted(Path("/kaggle/input").glob("**/*.zip")):   # dataset kept as a zipped folder
        target = SCRATCH / "code"
        shutil.rmtree(target, ignore_errors=True)
        shutil.unpack_archive(str(archive), str(target))
        for path in sorted(target.glob("**/scripts/train.py")):
            return path.parent.parent
    raise SystemExit("Attach the crosswise-ml-code dataset (kaggle_run.py does this for you).")


source = find_code()
if ML.exists():
    shutil.rmtree(ML)
shutil.copytree(source, ML)
for name in ("data", "runs"):     # heavy folders live in scratch: /kaggle/working is limited to 20 GB of output
    (SCRATCH / name).mkdir(parents=True, exist_ok=True)
    link = ML / name
    if link.exists() or link.is_symlink():
        shutil.rmtree(link, ignore_errors=True)
    link.symlink_to(SCRATCH / name, target_is_directory=True)
ARTIFACTS.mkdir(exist_ok=True)
os.chdir(ML)
sys.path.insert(0, str(ML))
print("mode:", MODE, "| code:", source, "| scratch:", SCRATCH)
print(sorted(p.name for p in ML.iterdir()))

## 1. Environment and unit tests

In [ ]:
run("pip install -q ultralytics ijson gdown roboflow")
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
if MODE == "full" and not torch.cuda.is_available():
    # On CPU this run would take a day and still hit Kaggle's 12 h limit, so stop while it costs nothing.
    raise SystemExit("No GPU: push with --gpu, and phone-verify the account at kaggle.com/settings.")
run("python -m pytest tests -q")

## 2. Smoke run
Synthetic scenes and a 12-second clip (red → green, with a car growing in the frame) exercise every script end to end:
dataset merge, teacher pseudo-labels, training, video auto-labeling with temporal voting, and the review round-trip.

In [ ]:
if MODE == "smoke":
    run("python scripts/make_smoke_dataset.py")
    run("python scripts/build_dataset.py --sources configs/sources_smoke.yaml --out data/merged --copy")
    run("python scripts/pseudo_label.py --data data/merged/data.yaml --teacher yolo26n.pt --imgsz 640")
    # 80 epochs on 24 toy images is ~4 min on CPU and enough for the model to find the flat colored lamps,
    # so the auto-labeling step below gets something real to work with.
    run("python scripts/train.py --data data/merged/data.yaml --name crosswise_smoke"
        " --epochs 80 --imgsz 320 --batch 8")

In [ ]:
if MODE == "smoke":
    run("python scripts/autolabel_video.py --videos data/videos"
        " --model runs/detect/crosswise_smoke/weights/best.pt --out data/raw/local_auto"
        " --every-ms 500 --imgsz 320 --conf 0.10 --teacher yolo26n.pt --open-vocab none")
    run("python scripts/review_labels.py make --dataset data/raw/local_auto")

    from collections import Counter
    from crosswise_ml.classes import CANONICAL
    from crosswise_ml.review import build_items
    from crosswise_ml.yolo_io import read_labels

    # What did the auto-labeler actually write? Empty label files would make the review round-trip below meaningless.
    counts = Counter(CANONICAL[box.cls] for path in Path("data/raw/local_auto/labels").rglob("*.txt")
                     for box in read_labels(path))
    print("auto-labeled boxes:", dict(counts) or "NONE")
    if not counts:
        from ultralytics import YOLO          # show the raw confidences, so a threshold problem is visible
        frames = sorted(Path("data/raw/local_auto/images").rglob("*.jpg"))
        frame = frames[len(frames) // 4]
        weights = "runs/detect/crosswise_smoke/weights/best.pt"
        result = YOLO(weights).predict(str(frame), imgsz=320, conf=0.001, verbose=False)[0]
        print(f"top raw detections on {frame.name}:")
        for cls, conf in sorted(zip(result.boxes.cls.tolist(), result.boxes.conf.tolist()), key=lambda p: -p[1])[:8]:
            print(f"   {result.names[int(cls)]:<10} {conf:.4f}")

    items = build_items(Path("data/raw/local_auto"))     # stand in for a human reviewer: accept every frame
    print("frames to review:", len(items), "| flagged:", sum(1 for item in items if item["flags"]))
    decisions = SCRATCH / "decisions.json"
    decisions.write_text(json.dumps({"decisions": {item["img"]: "accept" for item in items}}))
    run(f"python scripts/review_labels.py apply --dataset data/raw/local_auto --decisions {decisions}")
    if not (counts["ped_red"] or counts["ped_green"]):
        raise SystemExit("Auto-labeling found no pedestrian signals: the smoke run proved nothing")

## 3. Full run
Uses `configs/sources.yaml`. Roboflow needs the Kaggle secret `ROBOFLOW_API_KEY` (Add-ons → Secrets in the notebook editor).
Large datasets (VIDVIP, DTLD, Vistas) are attached as Kaggle datasets with `kaggle_run.py --dataset <user>/<slug>`; point
their `path:` entries at the mounted folder under `/kaggle/input/`.

In [ ]:
if MODE == "full":
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["ROBOFLOW_API_KEY"] = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        print("Roboflow key loaded from Kaggle secrets")
    except Exception as error:
        print("No ROBOFLOW_API_KEY secret; Roboflow sources will be skipped:", error)
    if not Path("configs/sources.yaml").exists():
        shutil.copy("configs/sources.example.yaml", "configs/sources.yaml")

    # Point each source at the dataset you attached: Kaggle mounts them somewhere under /kaggle/input, and the
    # exact path depends on the slug, so match on the source name instead of hard-coding it in sources.yaml.
    import yaml
    sources = yaml.safe_load(Path("configs/sources.yaml").read_text(encoding="utf-8"))
    mounted = [p.parent for p in Path("/kaggle/input").rglob("data.yaml")]
    print("attached datasets:", [str(p) for p in mounted] or "none")
    plain = lambda text: "".join(c for c in str(text).lower() if c.isalnum())   # vidvip-yolo ~ vidvip_yolo
    for source in sources.get("sources", []):
        if Path(source["path"]).exists():
            continue
        keys = {plain(source["name"]), plain(Path(source["path"]).name)}
        match = next((p for p in mounted if any(key in plain(p) for key in keys)), None)
        if match:
            print(f"  {source['name']}: {source['path']} -> {match}")
            source["path"] = str(match)
    Path("configs/sources.yaml").write_text(yaml.safe_dump(sources, sort_keys=False), encoding="utf-8")

    # Only ImVisible (~450 MB). Everything else is either attached as a dataset or, like Mapillary Vistas at
    # 21 GB, an opt-in download you would not want to discover halfway through a GPU run.
    run("python scripts/fetch_datasets.py --sources configs/sources.yaml --only imvisible", allow_fail=True)
    run("python scripts/build_dataset.py --sources configs/sources.yaml --out data/merged")
    run("python scripts/pseudo_label.py --data data/merged/data.yaml --teacher yolo26x.pt")

In [ ]:
if MODE == "full":
    # v0 exists only to mine ImVisible's weak labels. No ImVisible, no reason to train twice: that would burn
    # an hour of GPU quota to arrive at the same dataset.
    imvisible = Path("data/raw/imvisible")
    two_stage = imvisible.exists() and any(imvisible.rglob("*.csv"))
    print("ImVisible available:", two_stage)
    if two_stage:
        run(f"python scripts/train.py --data data/merged/data.yaml --name crosswise_v0"
            f" --epochs {CONFIG['epochs_v0']} --imgsz {CONFIG['imgsz']}")
        print("saved so far:", collect())
        # Mining is a bonus: if it fails, v1 still trains on what we already have.
        if run("python scripts/mine_imvisible.py --imvisible data/raw/imvisible"
               " --model runs/detect/crosswise_v0/weights/best.pt", allow_fail=True) == 0:
            run("python scripts/build_dataset.py --sources configs/sources.yaml --out data/merged")
            run("python scripts/pseudo_label.py --data data/merged/data.yaml --teacher yolo26x.pt")
    run(f"python scripts/train.py --data data/merged/data.yaml --name crosswise_v1"
        f" --epochs {CONFIG['epochs_v1']} --imgsz {CONFIG['imgsz']}")
    print("saved so far:", collect())
    run("python scripts/eval_imvisible.py --model runs/detect/crosswise_v1/weights/best.pt", allow_fail=True)

### Your own recordings
Auto-labeled with v1, then packed into `local_auto.zip` for you to review at home (A / F / R keys in `review.html`).
Put the `decisions.json` you download from that page into `ml/` and run again: this cell then applies it, rebuilds the
dataset with your reviewed frames as the **test** split and scores v1 on it. Your recordings never train the model, so
no second training run is needed.

In [ ]:
if MODE == "full":
    CLIP_SUFFIXES = {".mp4", ".mov", ".mkv", ".avi", ".m4v"}
    videos = Path("data/videos")
    clips = [p for p in videos.rglob("*") if p.suffix.lower() in CLIP_SUFFIXES] if videos.exists() else []
    if not clips:      # your recordings are usually an attached dataset, whatever its folder is called
        clips = [p for p in Path("/kaggle/input").rglob("*") if p.suffix.lower() in CLIP_SUFFIXES]
        videos = Path(os.path.commonpath([str(p) for p in clips])) if clips else videos
        if clips and videos.is_file():
            videos = videos.parent
    print(f"{len(clips)} clips in {videos}")
    decisions = Path("decisions.json")
    if clips:
        run(f"python scripts/autolabel_video.py --videos {videos} --imgsz {CONFIG['imgsz']}"
            f" --model runs/detect/crosswise_v1/weights/best.pt --out data/raw/local_auto")
        run("python scripts/review_labels.py make --dataset data/raw/local_auto")
        if decisions.exists():
            run(f"python scripts/review_labels.py apply --dataset data/raw/local_auto"
                f" --decisions {decisions} --drop-unreviewed")
            run("python scripts/build_dataset.py --sources configs/sources.yaml --out data/merged")
            run(f"python scripts/evaluate.py --model runs/detect/crosswise_v1/weights/best.pt"
                f" --data data/merged/data.yaml --imgsz {CONFIG['imgsz']}")
        else:
            print("No decisions.json in ml/: review local_auto.zip at home, put decisions.json in ml/, run again.")
        shutil.make_archive(str(ARTIFACTS / "local_auto"), "zip", "data/raw/local_auto")

## 4. LiteRT export
The step that cannot run on Windows. It exports the COCO baseline for the phone, plus the model trained above.

In [ ]:
run('pip install -q "ultralytics[export-litert]"')
run("python scripts/export_litert.py --weights yolo26n.pt --imgsz 640")
trained = sorted(glob.glob("runs/detect/*/weights/best.pt"))
if trained:
    run(f"python scripts/export_litert.py --weights {trained[-1]}"
        f" --imgsz {320 if MODE == 'smoke' else CONFIG['imgsz']}")

## 5. Collect artifacts (this is what `kaggle_run.py output` downloads)

In [ ]:
collected = collect()
print("\n".join(f"{p.name}  {p.stat().st_size / 1e6:.1f} MB" for p in sorted(ARTIFACTS.iterdir())))

expected = ["yolo26n.tflite", "pseudo_labels.csv", "crosswise_smoke_best.pt", "crosswise_smoke_best.tflite",
            "crosswise_smoke_crosswise_report.json", "local_auto_review_summary.json",
            "local_auto_autolabel_report.csv"] if MODE == "smoke" else []
missing = [name for name in expected if name not in collected]
shutil.rmtree(ML, ignore_errors=True)     # keep only the artifacts in the notebook output
if missing:                               # fail the kernel: a silent gap in the outputs is the bug worth catching
    raise SystemExit(f"Missing expected artifacts: {missing}")
print("\nOK")